In [1]:
!uv pip install langchain langchain-community langchain-core langchain-chroma langchain-huggingface langchain-ollama sentence-transformers pypdf ipywidgets pydantic-settings

Using Python 3.12.3 environment at: /mnt/gamer_d/eye_ai_workspace/venvs/eye_ai_venv_312
Audited 10 packages in 136ms


In [ ]:
import os
import json
import ollama
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError
from pydantic_settings import BaseSettings
from IPython.display import Markdown, display, JSON

# LangChain components
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import LongContextReorder

In [ ]:
class Settings(BaseSettings):
    """Configurações centralizadas com fail-fast."""
    VECTOR_DB_PATH: str = "./chroma_db_hermes"
    MODEL_NAME: str = "gemma4:e4b" # "gemma3:4b-it-qat"
    EMBEDDING_MODEL_NAME: str = "all-MiniLM-L6-v2"
    RAG_DOCS_DIR: str = "../docs/rag"
    CHUNK_SIZE: int = 800
    CHUNK_OVERLAP: int = 150
    RETRIEVAL_K: int = 5
    
    class Config:
        env_file = ".env"

settings = Settings()
print(f"[CONFIG] Modelo: {settings.MODEL_NAME} | DB: {settings.VECTOR_DB_PATH}")

[CONFIG] Modelo: gemma3:4b-it-qat | DB: ./chroma_db_hermes


/tmp/ipykernel_1976543/1200216665.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class Settings(BaseSettings):


In [25]:
class ClinicalResponse(BaseModel):
    """Entidade de Domínio para resposta clínica validada."""
    patologia: str = Field(..., description="Nome da doença ou síndrome e CID se disponível")
    decisao_clinica: str = Field(..., description="A conduta ou resposta baseada nos protocolos")
    justificativa: str = Field(..., description="Explicação extraída do texto citando termos do contexto")
    fonte: List[str] | str = Field(..., description="Documentos que sustentam a resposta")
    alerta_alucinacao: bool = Field(default=False, description="True se a informação não foi encontrada no contexto")

    @classmethod
    def from_llm_output(cls, raw_json: str) -> 'ClinicalResponse':
        try:
            data = json.loads(raw_json)
            return cls(**data)
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"[ERR] Erro de validação de domínio: {e}")
            # Retorno de fallback seguro
            return cls(
                patologia="Erro de Processamento",
                decisao_clinica="Não foi possível validar a resposta do modelo.",
                justificativa=f"O output do LLM foi inválido: {raw_json[:100]}...",
                fonte="N/A",
                alerta_alucinacao=True
            )


In [26]:
class VectorDBAdapter:
    """Adapter de Infraestrutura para o banco vetorial ChromaDB."""
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name=settings.EMBEDDING_MODEL_NAME)
        self.db = None

    def load_or_create(self, force_reingest: bool = False) -> Chroma:
        if os.path.exists(settings.VECTOR_DB_PATH) and not force_reingest:
            print(f"[DB] Carregando banco persistente: {settings.VECTOR_DB_PATH}")
            self.db = Chroma(persist_directory=settings.VECTOR_DB_PATH, embedding_function=self.embeddings)
        else:
            self.db = self._ingest()
        return self.db

    def _ingest(self) -> Chroma:
        print(f"[DB] Iniciando ingestão de: {settings.RAG_DOCS_DIR}")
        loader = DirectoryLoader(settings.RAG_DOCS_DIR, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
        documentos = loader.load()
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.CHUNK_SIZE, 
            chunk_overlap=settings.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        chunks = splitter.split_documents(documentos)

        # [RESET SEGURO] Deleta a coleção via API em vez de apagar a pasta (evita locks)
        if os.path.exists(settings.VECTOR_DB_PATH):
            temp_db = Chroma(persist_directory=settings.VECTOR_DB_PATH, embedding_function=self.embeddings)
            try: temp_db.delete_collection()
            except: pass
            
        db = Chroma.from_documents(chunks, self.embeddings, persist_directory=settings.VECTOR_DB_PATH)
        print(f"[DB] Ingestão concluída ({len(chunks)} chunks).")
        return db

    def retrieve(self, query: str) -> List:
        if not self.db: self.load_or_create()
        
        # Busca os Top K
        retriever = self.db.as_retriever(search_kwargs={"k": settings.RETRIEVAL_K})
        docs_relevantes = retriever.invoke(query)
        
        # [ARQUITETURA] Scored Reorder para mitigar "Lost in the Middle"
        reorder = LongContextReorder()
        docs_reordenados = reorder.transform_documents(docs_relevantes)
        
        return docs_reordenados


In [27]:
class ClinicalLLMAdapter:
    """Adapter para o motor de inferência Ollama."""
    def __init__(self):
        self.llm = OllamaLLM(model=settings.MODEL_NAME, format="json", temperature=0.0)
        self.prompt_template = PromptTemplate.from_template("""
            Você é o Visio-Chat Hermes, um sistema de suporte à decisão clínica com RIGOR ABSOLUTO.
            Sua única fonte de verdade é o CONTEXTO INSTITUCIONAL fornecido abaixo.
            
            REGRAS CRÍTICAS:
            1. Se a resposta não estiver EXPLICITAMENTE em uma das fontes abaixo, defina "alerta_alucinacao": true.
            2. Se você encontrar a resposta, cite o nome da [FONTE] na sua justificativa.
            3. IGNORE seu conhecimento médico prévio se ele não estiver suportado pelo texto fornecido.
            4. Não faça deduções heróicas. Se o texto diz "A causa B", responda isso.

            CONTEXTO INSTITUCIONAL:
            {contexto}

            PERGUNTA DO MÉDICO:
            {pergunta}

            Responda OBRIGATORIAMENTE em formato JSON:
            - "patologia": Título da condição e CID.
            - "decisao_clinica": Conduta terapêutica ou diagnóstica imediata.
            - "justificativa": Raciocínio clínico citando a [FONTE] específica.
            - "fonte": Nomes dos arquivos de origem consultados.
            - "alerta_alucinacao": Booleano (true/false).

            Retorne APENAS o JSON.
            """)

    def generate(self, pergunta: str, contexto: str) -> str:
        chain = self.prompt_template | self.llm
        return chain.invoke({"contexto": contexto, "pergunta": pergunta})

In [31]:
class VisioChatHermes:
    def __init__(self):
        self.vector_db = VectorDBAdapter()
        self.llm = ClinicalLLMAdapter()

    def ask(self, pergunta: str) -> (ClinicalResponse, List):
        print(f"[VISIO-CHAT HERMES] Analisando consulta: '{pergunta}'")
        
        # 1. Recuperação
        docs = self.vector_db.retrieve(pergunta)
        
        # Formatação do contexto com metadados (Strict Grounding)
        contexto_formatado = []
        for doc in docs:
            fonte = os.path.basename(doc.metadata.get('source', 'Desconhecida'))
            contexto_formatado.append(f"[FONTE: {fonte}]\n{doc.page_content}")
        
        contexto = "\n\n---\n\n".join(contexto_formatado)
        
        # 2. Geração
        raw_output = self.llm.generate(pergunta, contexto)
        
        # 3. Validação de Domínio
        response = ClinicalResponse.from_llm_output(raw_output)
        
        return response, docs


In [32]:
from IPython.display import Markdown, display, HTML

def renderizar_dashboard_hermes(pergunta: str, output_json: str, docs_relevantes=None):
    """
    Renderiza um dashboard clínico elegante no Jupyter Notebook.
    """
    try:
        dados = json.loads(output_json)
    except:
        print("Erro ao processar JSON:", output_json)
        return

    # Cores e Estilos
    alerta = dados.get("alerta_alucinacao", False)
    status_color = "#ef4444" if alerta else "#22c55e"
    status_text = "⚠️ ALERTA: DADOS NÃO ENCONTRADOS" if alerta else ""
    
    html_header = f"""
    <div style=\"font-family: sans-serif; border: 1px solid #e5e7eb; border-radius: 8px; overflow: hidden; margin-bottom: 20px;\">
        <div style=\"background-color: {status_color}; color: white; padding: 12px 20px; font-weight: bold; display: flex; justify-content: space-between;\">
            <span>VISIO-CHAT HERMES | Suporte à Decisão</span>
            <span>{status_text}</span>
        </div>
        <div style=\"padding: 20px; background-color: #f9fafb;\">
            <h3 style=\"margin-top: 0; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 8px;\">❓ Pergunta do Médico</h3>
            <p style=\"font-size: 1.1em; color: #374151; font-weight: 500; line-height: 1.6;\">{pergunta}</p>

            <h3 style=\"margin-top: 24px; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 8px;\">🎯 Decisão Clínica</h3>
            <p style=\"font-size: 1.1em; color: #374151; line-height: 1.6;\">{dados.get('decisao_clinica', 'N/A')}</p>
            
            <h3 style=\"margin-top: 24px; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 8px;\">📖 Justificativa e Evidências</h3>
            <p style=\"color: #4b5563; line-height: 1.6; font-style: italic;\">{dados.get('justificativa', 'N/A')}</p>
        </div>
    </div>
    """
    
    display(HTML(html_header))

    # Se houver documentos, mostrar as fontes de forma resumida
    if docs_relevantes:
        fontes_html = "<div style='font-family: sans-serif; padding: 0 10px;'><h4>📄 Fontes Institucionais Consultadas:</h4><ul style='color: #6b7280; font-size: 0.9em;'>"
        fontes_unicas = set()
        for doc in docs_relevantes:
            nome_arquivo = os.path.basename(doc.metadata.get('source', 'Desconhecido'))
            fontes_unicas.add(nome_arquivo)
        
        for fonte in fontes_unicas:
            fontes_html += f"<li>{fonte}</li>"
        
        fontes_html += "</ul></div>"
        display(HTML(fontes_html))


In [33]:
# ==========================================
# EXECUÇÃO VIA FACHADA DE APLICAÇÃO
# ==========================================
if __name__ == "__main__":
    # Instancia o Consultor (Application Service)
    visio_chat_hermes = VisioChatHermes()
    
    # Opcional: Forçar reingestão se houver novos arquivos
    # visio_chat_hermes.vector_db.load_or_create(force_reingest=False)

    # Consulta de Teste
    pergunta = "Paciente com baixa de visão e catarata, além de disúria. Qual pode ser o diagnóstico?"
    resposta, docs_originais = visio_chat_hermes.ask(pergunta)

    # Renderização
    renderizar_dashboard_hermes(pergunta, resposta.model_dump_json(), docs_originais)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[VISIO-CHAT HERMES] Analisando consulta: 'Paciente com baixa de visão e catarata, além de disúria. Qual pode ser o diagnóstico?'
[DB] Carregando banco persistente: ./chroma_db_hermes


In [ ]:
import chromadb
import pandas as pd

def inspecionar_colecoes():
    """Ferramenta de DX para visualizar o estado interno do banco de vetores."""
    client = chromadb.PersistentClient(path=settings.VECTOR_DB_PATH)
    collections = client.list_collections()
    
    print(f"[INSPECTOR] Localizado: {settings.VECTOR_DB_PATH}")
    for col in collections:
        count = col.count()
        print(f"\nColeção: '{col.name}' | Total de Chunks: {count}")
        
        if count > 0:
            dados = col.get(limit=5)
            df = pd.DataFrame({
                'ID': dados['ids'],
                'Documento': [d[:150] + '...' for d in dados['documents']],
                'Metadata': dados['metadatas']
            })
            display(df)

inspecionar_colecoes()